In [1]:
#!/usr/bin/env python3

import pickle
import networkx as nx
from collections import deque
import pandas as pd
import random

# ============================================================
# PARAMETERS
# ============================================================

INPUT_EDGELIST = "Databases/merged.csv"         
INPUT_GRAPH_PKL = None               

OUTPUT_PATHS = "ppi_disease_paths.pkl"
OUTPUT_SUMMARY = "ppi_disease_summary.csv"
OUTPUT_RANDOM = "random_baseline.pkl"

MAX_DEPTH = 5
N_RANDOM_SAMPLES = 200

DIRECTED = False 

# ============================================================
# SEED PROTEINS (disease module)
# ============================================================


SEED_PROTEINS = set(["P45985", "Q14315"
])

# ============================================================
# LOAD GRAPH
# ============================================================

print("Loading PPI graph...")

if INPUT_GRAPH_PKL:
    with open(INPUT_GRAPH_PKL, "rb") as f:
        G = pickle.load(f)
else:
    df_edges = pd.read_csv(INPUT_EDGELIST)
    GraphType = nx.DiGraph if DIRECTED else nx.Graph
    G = nx.from_pandas_edgelist(
        df_edges,
        source="Uniprot_A",
        target="Uniprot_B",
        edge_attr=True,         
        create_using=GraphType(),
    )

print("Graph loaded")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# ============================================================
# SEED NODES PRESENT IN GRAPH
# ============================================================

seed_nodes = [n for n in SEED_PROTEINS if n in G]
print("Disease/seed nodes found:", len(seed_nodes))

if not seed_nodes:
    raise ValueError("No seed proteins found in the graph — check SEED_PROTEINS UniProt IDs against your node IDs.")

# ============================================================
# ALL PROTEIN NODES
# ============================================================

protein_nodes = list(G.nodes())
print("Total proteins:", len(protein_nodes))

# ============================================================
# DISTANCE TO DISEASE MODULE (Barabási-style)
# ============================================================

print("Precomputing distances to disease module...")

min_dist_to_S = {}

for s in seed_nodes:
    lengths = nx.single_source_shortest_path_length(G, s, cutoff=MAX_DEPTH)
    for node, d in lengths.items():
        if node not in min_dist_to_S or d < min_dist_to_S[node]:
            min_dist_to_S[node] = d

# ============================================================
# RANDOM BASELINE
# ============================================================

print("Computing random baseline...")

random_baseline = {}

for size in range(1, 21):
    samples = []
    for _ in range(N_RANDOM_SAMPLES):
        sample = random.sample(protein_nodes, size)
        d_vals = [min_dist_to_S.get(t, MAX_DEPTH + 1) for t in sample]
        samples.append(sum(d_vals) / len(d_vals))
    random_baseline[size] = samples

print("Random baseline ready.")

# ============================================================
# BFS: FULL PATHS FROM SEED PROTEINS TO ALL REACHABLE PROTEINS
# ============================================================

print("Starting BFS path search...")

queue = deque()
visited = {}

for protein in seed_nodes:
    queue.append((protein, [protein], 0))
    visited[protein] = 0

paths_found = []

while queue:
    node, path, depth = queue.popleft()

    if depth >= MAX_DEPTH:
        continue

    neighbors = G.successors(node) if DIRECTED else G.neighbors(node)

    for nxt in neighbors:
        new_depth = depth + 1
        new_path = path + [nxt]

        if nxt not in visited or visited[nxt] > new_depth:
            visited[nxt] = new_depth
            queue.append((nxt, new_path, new_depth))

            edge_data = G.get_edge_data(node, nxt) or {}

            paths_found.append({
                "seed": path[0],
                "target": nxt,
                "path_nodes": new_path,
                "path_length": new_depth,
                "last_edge_attrs": edge_data,
            })

print("Paths found:", len(paths_found))

# ============================================================
# SAVE PATH DATA
# ============================================================

print("Saving path data...")

with open(OUTPUT_PATHS, "wb") as f:
    pickle.dump(paths_found, f)

with open(OUTPUT_RANDOM, "wb") as f:
    pickle.dump(random_baseline, f)

# ============================================================
# SUMMARY
# ============================================================

print("Creating summary...")

df = pd.DataFrame(paths_found)

summary = df.groupby("seed").agg(
    targets_reached=("target", "nunique"),
    paths=("target", "count"),
    max_path_length=("path_length", "max"),
).reset_index()

summary.to_csv(OUTPUT_SUMMARY, index=False)

print("Pipeline finished.")

Loading PPI graph...
Graph loaded
Nodes: 38017
Edges: 1161480
Disease/seed nodes found: 2
Total proteins: 38017
Precomputing distances to disease module...
Computing random baseline...
Random baseline ready.
Starting BFS path search...
Paths found: 37819
Saving path data...
Creating summary...
Pipeline finished.


In [ ]:
# ============================================================
# COMPUTE DRUG PROXIMITY Z-SCORES
# ============================================================

import ast
import numpy as np
import pandas as pd
import os

# ============================================================
# INPUT
# ============================================================

INPUT_FILE = (
    "drug_level_statistics.csv"
)

BASELINE_FILE = (
    "/content/drive/MyDrive/INDRA/improved_random_2/"
    "baseline_mu_sigma.csv"
)

OUTPUT_FILE = (
    "drug_z_scores.csv"
)

os.makedirs(
    os.path.dirname(OUTPUT_FILE),
    exist_ok=True
)

# ============================================================
# LOAD DATA
# ============================================================

drug_df = pd.read_csv(INPUT_FILE)

baseline_df = pd.read_csv(BASELINE_FILE)

baseline = {

    int(row["k"]): (

        row["mu_k"],

        row["sigma_k"]

    )

    for _, row in baseline_df.iterrows()

}

# ============================================================
# COMPUTE Z-SCORES
# ============================================================

rows = []

for _, row in drug_df.iterrows():

    k = int(row["k"])

    if k not in baseline:

        continue

    target_distances = ast.literal_eval(
        row["target_distances"]
    )

    distances = list(
        target_distances.values()
    )

    d_drug = np.mean(distances)

    mu_k, sigma_k = baseline[k]

    if sigma_k == 0:

        continue

    z_score = (

        d_drug - mu_k

    ) / sigma_k

    rows.append({

        "drug": row["drug"],

        "k": k,

        "d_drug": d_drug,

        "mu_k": mu_k,

        "sigma_k": sigma_k,

        "z_score": z_score,

        "path_count": row["path_count"],

        "shortest_path": row["shortest_path"],

        "avg_belief": row["avg_belief"],

        "avg_corr_weight": row["avg_corr_weight"]

    })

# ============================================================
# SAVE
# ============================================================

z_df = pd.DataFrame(rows)

z_df = z_df.sort_values(

    "z_score",

    ascending=True

)

z_df.to_csv(

    OUTPUT_FILE,

    index=False

)

# ============================================================
# SUMMARY
# ============================================================

print()

print("=" * 60)

print("DRUG PROXIMITY Z-SCORES")

print("=" * 60)

print()

print("Drugs analysed:", len(z_df))

print()

print("Saved:")

print(OUTPUT_FILE)

print()

display(

    z_df.head(20)


In [5]:
import pickle

# Full path records
with open("ppi_disease_paths.pkl", "rb") as f:
    paths_found = pickle.load(f)

print(len(paths_found))  
print(paths_found[0])   

# Random baseline
with open("random_baseline.pkl", "rb") as f:
    random_baseline = pickle.load(f)

print(random_baseline[4]) 
import pandas as pd

df = pd.DataFrame(paths_found)
print(df.head())
print(df.columns.tolist())

37819
{'seed': 'P45985', 'target': 'Q8BTM8', 'path_nodes': ['P45985', 'Q8BTM8'], 'path_length': 1, 'last_edge_attrs': {'source_file': 'Databases\\BioGRID\\sanitised.csv'}}
[2.75, 2.5, 2.0, 2.5, 2.75, 2.0, 2.25, 2.5, 2.5, 2.5, 2.5, 2.75, 2.5, 2.0, 2.5, 2.25, 2.5, 2.25, 2.0, 2.25, 2.5, 2.25, 2.75, 2.25, 2.75, 2.25, 2.25, 3.25, 2.25, 2.75, 2.75, 2.5, 2.0, 2.5, 2.75, 2.75, 2.75, 2.25, 2.75, 2.5, 2.25, 2.0, 2.5, 2.5, 3.0, 3.25, 2.5, 2.25, 2.25, 2.25, 2.25, 2.5, 2.0, 2.5, 2.75, 2.5, 2.5, 2.25, 3.0, 2.75, 2.5, 2.25, 3.0, 2.5, 2.75, 2.25, 2.5, 2.25, 2.75, 2.5, 2.5, 3.0, 2.5, 2.25, 2.5, 3.0, 2.75, 2.25, 2.5, 3.0, 2.25, 2.25, 2.75, 2.5, 2.25, 2.5, 2.25, 2.5, 2.25, 2.5, 2.25, 2.5, 2.75, 2.75, 2.25, 2.5, 2.0, 2.5, 2.5, 2.75, 3.0, 2.25, 2.5, 2.25, 2.75, 2.25, 2.25, 2.75, 2.75, 2.75, 2.5, 2.5, 2.25, 2.75, 3.25, 2.75, 2.5, 3.0, 2.5, 2.25, 3.25, 3.75, 2.5, 2.75, 3.75, 2.5, 2.75, 2.5, 2.25, 3.25, 2.75, 2.75, 2.75, 2.25, 3.0, 2.5, 2.75, 3.0, 2.0, 2.25, 2.25, 2.5, 3.0, 3.25, 2.75, 2.5, 2.5, 2.5, 2.5, 2.7